# Module 4.1: The Encoder Layer

We've built all the raw materials: Tensors, Embeddings, Positional Encoding, and Multi-Head Attention. Now, it's time to build the core architecture block. 

A full Transformer (like BERT) is just 12 or 24 of these **Encoder Blocks** stacked directly on top of each other. But we can't just stack Attention on top of Attention; the math would explode. We need stabilizers.

> ⚠️ **Read this first — what we are building and why.**
>
> Modules **4.1–4.3** build the *historical* **2017 Encoder–Decoder Transformer**, because building both halves is the clearest way to learn each component in isolation.
>
> From **Module 4.2 onward we keep ONLY the Decoder stack** — the Llama-style, *decoder-only* design used by every modern chat model (GPT, Llama, Claude). In particular, **Cross-Attention will be deleted**. Learn it here for understanding the original architecture, *not* because the final model uses it.

## 1. The Missing Pieces

### The Concept
Attention is incredibly powerful for *finding context*, but it has two major flaws:
1. It is mathematically unstable. Repeated matrix multiplications will cause numbers to vanish to zero or explode to infinity.
2. It doesn't actually "think." Attention just moves vectors around so they communicate. It doesn't process that newly combined information.

### Why do we need stabilizers?
To make deep neural networks trainable, we need to guarantee that the gradients during Backpropagation flow smoothly back to the first layer, and we need dedicated layers for memorizing facts.

## 2. Residual Connections (The Highway)

### The Analogy (The Editor)
Imagine you write a draft of an essay. You give it to an editor (the Attention Network). The editor completely rewrites the essay from scratch. Sometimes they make it better, but sometimes they ruin everything you wrote! 

Instead, what if the editor just hands you a list of *edits* (the residuals), and you simply **add** those edits to your original draft? If the edits are bad, you can easily ignore them and keep the original.

### The Math
$$ Output = Input + \text{Sublayer}(Input) $$
In code, this is literally just `out = x + layer(x)`.

### Why do we need it?
This creates an uninterrupted "highway" bypassing the heavy Matrix Multiplication. During Backpropagation, the gradient can sprint down this highway all the way to the Embeddings, completely solving the **Vanishing Gradient Problem**. It is arguably the most important trick in modern Deep Learning.

## 3. Layer Normalization (The Equalizer) vs RMSNorm

### The Concept
When you multiply matrices thousands of times, some numbers get huge (e.g., `4502`) and some get tiny (e.g., `-0.0001`). The huge numbers will mathematically drown out the tiny numbers completely. 

**Layer Normalization** fixes this by calculating the mean (average) *and* variance of the vector, then forcing the numbers to have a mean of **0** and a variance of **1**. It both re-centers and re-scales — like equalizing the sound levels.

### Modern Upgrade: RMSNorm
The original 2017 Transformer used standard LayerNorm. Modern LLMs (like Llama and Mistral) use **RMSNorm**, which **skips the mean entirely**. 

This is a key difference, so read carefully:
- **LayerNorm** subtracts the mean (re-centers to 0) *and* divides by the standard deviation (re-scales).
- **RMSNorm** only divides by the *root-mean-square* magnitude (re-scales). It does **NOT** subtract the mean, so the output is **not** centered at 0.

Researchers have found that dropping the mean-centering makes the math roughly 10–20% faster with little to no drop in quality.

### ⚠️ What to expect from the numbers below
Because our `RMSNorm` does **not** re-center, do **not** expect the output to have mean 0. The input `[-500, 2, 1000]` has a large positive average, so the *output* will also lean positive. What RMSNorm guarantees is that the overall *magnitude* is tamed (the root-mean-square of the output is ~1, scaled by the learned weight), turning wild values like `1000` into small, stable numbers.

### Why do we need it?
Without Normalization, the numbers spin out of control and the loss becomes `NaN` (Not a Number) during training.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)  # Reproducibility: same random numbers every run

# Let's write a simple RMSNorm (used in Llama 3!)
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        # The model can learn to scale the numbers up or down if it wants to!
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # Calculate the Root Mean Square
        rms = torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps)
        
        # Normalize by dividing by RMS, then multiply by learned weights.
        # NOTE: we never subtract the mean -> the output is NOT centered at 0.
        return (x / rms) * self.weight

dummy_input = torch.tensor([[-500.0, 2.0, 1000.0]])
norm = RMSNorm(dim=3)
output = norm(dummy_input)
print(f"Crazy Input:  {dummy_input[0]}")
print(f"Tamed Output: {output[0].detach()}")

# Proof that RMSNorm does NOT zero the mean (unlike LayerNorm):
print(f"\nMean of output: {output[0].mean().item():.4f}  <- NOT 0 (RMSNorm skips centering)")
print(f"RMS  of output: {output[0].pow(2).mean().sqrt().item():.4f}  <- ~1 (magnitude is tamed)")

## 4. The Feed-Forward Network (The Thinker)

### The Concept
After words have exchanged context through Attention (e.g., "bank" realizes it's near "river"), that new information needs to be processed. The Feed-Forward Network (FFN) is a standard neural network: it expands the dimension by 4x, runs a non-linear activation, and shrinks it back down to the original size.

> **Why shrink back?** The residual connection adds the FFN output back to the input (`x + ffn(x)`), so the output *must* have the same dimension as the input (`d_model`). That is exactly why we expand 4× internally but always return to `d_model`.

### A note on the activation function
The choice of activation has evolved over time, and this course shows a few:
- The **2017 paper** used `ReLU`.
- This notebook uses `GELU` (a smoother ReLU, common in GPT-2/BERT-era models).
- The Decoder Layer (Module 4.2) will use `ReLU` again just to keep that cell short.
- **Modern Llama-style models use `SwiGLU`**, a gated variant. We stick with `GELU`/`ReLU` here purely for simplicity — the architectural idea (expand → activate → shrink) is identical.

### Why do we need it?
Attention is responsible for *routing* information around the sentence; the FFN then *processes* it. Researchers believe a large fraction of the model's factual knowledge (e.g., "Paris is in France") is stored in these FFN weights — the 4× expansion gives it a large number of parameters to do so.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, hidden_dim: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden_dim)
        self.w2 = nn.Linear(hidden_dim, d_model)
        self.activation = nn.GELU()  # GELU here; the 2017 paper used ReLU, modern Llama uses SwiGLU

    def forward(self, x):
        # Expand, Activate, Shrink
        return self.w2(self.activation(self.w1(x)))

### Modern Upgrade: SwiGLU (build it — your capstone model uses this one)

The concept above never changes: *expand, non-linearity, shrink back*. What modern Llama-style models change is the middle step. **SwiGLU** runs **two** parallel up-projections: one passes through a smooth activation (SiLU) and then **gates** the other by element-wise multiplication — each hidden unit gets a learned, input-dependent "volume knob":

$$\text{SwiGLU}(x) = W_2\,\big(\mathrm{SiLU}(W_1 x) \odot W_3 x\big)$$

Why bother? Empirically it just trains better — enough that Llama, Mistral, PaLM and friends all switched. One bookkeeping consequence: SwiGLU has **three** matrices instead of two, so frameworks shrink the hidden width by ~⅓ (to $\tfrac{2}{3}\cdot 4d$) to keep the parameter count comparable.

This is the exact FFN inside `src/llm_workout/layers.py` — the one your capstone model will train in Module 5.3.

In [ ]:
import torch.nn.functional as F

class SwiGLU(nn.Module):
    """The Llama-style gated FFN -- identical to src/llm_workout/layers.py."""
    def __init__(self, d_model: int, hidden_dim: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)   # up-projection (gets gated)
        self.w2 = nn.Linear(hidden_dim, d_model, bias=False)   # down-projection
        self.w3 = nn.Linear(d_model, hidden_dim, bias=False)   # the gate's input

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))        # gate ⊙ value, then shrink

d_model = 128
count = lambda m: sum(p.numel() for p in m.parameters())

gelu_ffn = FeedForward(d_model, 4 * d_model)
swiglu_naive = SwiGLU(d_model, 4 * d_model)                # same width -> +50% params!
swiglu_matched = SwiGLU(d_model, int(4 * d_model * 2 / 3)) # the 2/3 trick -> comparable

print(f"GELU FFN   (hidden 4d):        {count(gelu_ffn):,} params")
print(f"SwiGLU     (hidden 4d):        {count(swiglu_naive):,} params  <- 3 matrices!")
print(f"SwiGLU     (hidden (2/3)*4d):  {count(swiglu_matched):,} params  <- the standard fix")

x = torch.randn(1, 5, d_model)
print(f"\nShape check: in {tuple(x.shape)} -> out {tuple(swiglu_matched(x).shape)} (unchanged, as the residual requires)")

## 5. Building the Full Encoder Block (Pre-Norm)

### The Concept
We have everything! Let's combine them.

> **Note**: The 2017 paper did Normalization *after* Attention (`x = Norm(x + Attention(x))`). 
> We will use the modern **Pre-Norm** standard (`x = x + Attention(Norm(x))`), which makes training significantly more stable.

> **Nothing here gets thrown away.** From Module 4.2 onward we keep only the *decoder* stack — but a decoder block is ~90% this exact block: same pre-norm layout, same residuals, same RMSNorm, same FFN. The only change is that its attention wears a causal mask.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../"))
# We MOCK MultiHeadAttention here so this block runs standalone and cleanly.
# A "mock" means we replace real attention with a single Linear layer that just
# reshuffles each vector. It has the right input/output shape, but it does NOT
# actually do the Query/Key/Value multi-head split.
#   -> For the REAL multi-head attention (splitting d_model into num_heads heads,
#      computing scaled dot-product per head, then concatenating), see Notebook 08.
class MockMultiHeadAttention(nn.Module):
    def __init__(self, d_model): 
        super().__init__()
        self.proj = nn.Linear(d_model, d_model)
    def forward(self, x): 
        return self.proj(x)

class TransformerEncoderBlock(nn.Module):
    # NOTE: `num_heads` is accepted to match the real signature, but it is UNUSED
    # here because our attention is mocked (see Notebook 08 for where it matters).
    def __init__(self, d_model: int, num_heads: int, hidden_dim: int):
        super().__init__()
        
        # 1. The Attention Block
        self.attention = MockMultiHeadAttention(d_model)
        self.norm1 = RMSNorm(d_model)
        
        # 2. The Thought Block
        self.ffn = FeedForward(d_model, hidden_dim)
        self.norm2 = RMSNorm(d_model)

    def forward(self, x):
        # --- Step 1: Attention --- 
        # Modern Pre-Norm Architecture with Residual Highway
        x = x + self.attention(self.norm1(x))
        
        # --- Step 2: Feed Forward ---
        # Another Pre-Norm and Residual Highway
        x = x + self.ffn(self.norm2(x))
        
        return x

# --- TEST TIME ---
# Pretend we have a batch of 2 sentences, 5 words each, mapped to 128 dimensions
batch_size, seq_len, d_model = 2, 5, 128

# The "4x rule": the FFN hidden dimension is conventionally 4 * d_model.
hidden_dim = 4 * d_model
print(f"d_model = {d_model}, so hidden_dim = 4 * {d_model} = {hidden_dim}")

dummy_embeds = torch.randn(batch_size, seq_len, d_model)

encoder_layer = TransformerEncoderBlock(d_model=d_model, num_heads=8, hidden_dim=hidden_dim)

output = encoder_layer(dummy_embeds)

print(f"\nInput Shape:  {dummy_embeds.shape}")
print(f"Output Shape: {output.shape} (Dimensions stay perfectly intact!)")

### Seeing the instability for ourselves (why we need the stabilizers)

At the very top we claimed that "stacking Attention on top of Attention makes the math explode." Our mocked attention is just a `Linear` layer, so it can't fully reproduce real attention's instability — but we *can* show the core idea: repeatedly pushing a vector through many sublayers **without** any normalization lets the magnitude drift far from 1, while a normalization step after each step keeps it tame.

In [ ]:
# Stack many mocked sublayers and watch the magnitude with vs. without normalization.
torch.manual_seed(0)
x_no_norm = torch.randn(1, 128)
x_with_norm = x_no_norm.clone()

layers = [nn.Linear(128, 128) for _ in range(40)]
normer = RMSNorm(128)

print("layer |  no norm (RMS)  |  with norm (RMS)")
for i, layer in enumerate(layers):
    x_no_norm = layer(x_no_norm)
    x_with_norm = normer(layer(x_with_norm))  # normalize after each sublayer
    if i % 8 == 0 or i == len(layers) - 1:
        rms_no = x_no_norm.detach().pow(2).mean().sqrt().item()
        rms_yes = x_with_norm.detach().pow(2).mean().sqrt().item()
        print(f"{i:5d} | {rms_no:14.4f} | {rms_yes:14.4f}")

print("\nThe un-normalized magnitude drifts away from 1; the normalized one stays controlled.")

### 🏋️ Try it yourself

**Task 1 — Kill the residual.** Make a copy of `TransformerEncoderBlock` where the `forward` method drops the `x +` part (i.e. `x = self.attention(self.norm1(x))` instead of `x = x + ...`). Stack ~20 of these blocks, run a random input through, and compare the output to the residual version. Then think: during *training*, why would removing `x +` make the gradient struggle to reach the early layers?

**Task 2 — Change the FFN width.** Re-run the block with `hidden_dim = 2 * d_model` and `hidden_dim = 8 * d_model`. The output shape should be unchanged (why?). Roughly how does the FFN parameter count scale with `hidden_dim`?

In [ ]:
# Task 1 starter: an encoder block WITHOUT the residual connection.
class NoResidualEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, hidden_dim):
        super().__init__()
        self.attention = MockMultiHeadAttention(d_model)
        self.norm1 = RMSNorm(d_model)
        self.ffn = FeedForward(d_model, hidden_dim)
        self.norm2 = RMSNorm(d_model)

    def forward(self, x):
        # TODO: remove the "x +" below and observe what changes.
        x = x + self.attention(self.norm1(x))   # <-- try: x = self.attention(self.norm1(x))
        x = x + self.ffn(self.norm2(x))          # <-- try: x = self.ffn(self.norm2(x))
        return x

# Your experiment here:
torch.manual_seed(0)
sample = torch.randn(1, 5, 128)
block = NoResidualEncoderBlock(128, 8, 4 * 128)
print("Output shape:", block(sample).shape)

# Task 2 starter:
# for h in (2 * 128, 4 * 128, 8 * 128):
#     ff = FeedForward(128, h)
#     n_params = sum(p.numel() for p in ff.parameters())
#     print(f"hidden_dim={h}: {n_params:,} params")